# 02 Flickr activity labeling (LEGACY)

Extracts per-image labels from the old Qwen3-VL run and maps them to the LVVI3 taxonomy.
Superseded by `scripts/Flickr/multimodal_analysis/run_clef_flickr.py`.
`sub_activity` is kept only at the preprocessing level.

**Inputs:**
- `data/_archive/old_llm_subactivity/flickr/inference_json_output/` — old Qwen3-VL per-image JSON
  outputs (archived; the multimodal pipeline and its frame paths are gone)

**Outputs:**
- `data/_archive/old_llm_subactivity/flickr/Flickr_inferenced.csv` — per-image activity /
  sub_activity labels joined to the LVVI3 taxonomy (`activity`, `sub_activity`, `type`)

**Position:** Step 02 of Flickr preprocessing (`scripts/Flickr/data_preprocessing/`).
Downstream: `scripts/RQs_analysis/trip_identification.ipynb` reads the output CSV.

> **NOTE:** `data/LVVI3/LVVI3_activities_inference_postprocessing.csv` **no longer exists**
> (it lived in the deleted LVVI3_YouTube project). The cell below that reads it must be
> **reconstructed before this notebook is re-run**; the rest of the notebook is unaffected.

# Load packages

In [1]:
import pandas as pd
import geopandas as gpd

import json
from pathlib import Path
from tqdm import tqdm

# Clean activities

## Read LLM inferenced information from JSON files 

In [4]:
# Folder containing LLM inferenced information JSON files
json_folder = Path("../../../data/_archive/old_llm_subactivity/flickr/inference_json_output")

json_files = list(json_folder.glob("*.json"))

rows = []

for file in tqdm(json_files, desc="Processing JSON files"):

    try:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        image_id = data.get("image_id")

        base_row = {
            "image_id": image_id,
            "human_presence": data.get("human_presence"),
            "accompany_type": data.get("accompany_type", "").strip(),
            "age_group": data.get("age_group", "").strip(),
            "gender": data.get("gender", "").strip(),
        }

        activities = data.get("activities", [])

        # keep rows even without activities
        if not activities:
            rows.append({
                **base_row,
                "activity": None,
                "type": None
            })

        else:
            for activity in activities:

                activity_name = activity.get("name", "").strip()

                if activity_name:
                    activity_name = (
                        activity_name[:1].upper() +
                        activity_name[1:].lower()
                    )

                rows.append({
                    **base_row,
                    "activity": activity_name,
                    "type": activity.get("type", "").strip()
                })

    except Exception as e:
        print(f"\nError reading {file.name}: {e}")

df_inferenced = (
    pd.DataFrame(rows)
    .drop_duplicates(ignore_index=True)
)

df_inferenced.to_csv('../../../data/_archive/old_llm_subactivity/flickr/Flickr_inferenced.csv', index=False)

print(df_inferenced.shape)

Processing JSON files: 0it [00:00, ?it/s]

(0, 0)


In [12]:
df_inferenced = pd.read_csv('../../../data/_archive/old_llm_subactivity/flickr/Flickr_inferenced.csv')
df_inferenced = df_inferenced[~df_inferenced['activity'].isna()]
df_inferenced.shape

(1775, 7)

In [13]:
df_inferenced

,image_id,human_presence,accompany_type,age_group,gender,activity,type
13,48104010326,1,Unknown,Unknown,Unknown,"Forestry work such as planting, nursery manage...",survey_activity
14,48032028188,1,Unknown,Unknown,Unknown,Diving,survey_activity
15,48032028188,1,Unknown,Unknown,Unknown,Scuba diving,survey_activity
16,48032028188,1,Unknown,Unknown,Unknown,Underwater photography,new_activity
18,49538232706,1,Unknown,Unknown,Unknown,Mountain biking,survey_activity
...,...,...,...,...,...,...,...
17021,50628547668,1,Unknown,Unknown,Unknown,Boating,survey_activity
17031,51049875966,1,Unknown,Unknown,Unknown,Cycling,survey_activity
17050,51881493691,1,Unknown,Unknown,Unknown,Snowshoeing,survey_activity
17051,51881493691,1,Unknown,Unknown,Unknown,Ice fishing,survey_activity


## Read LVVI3 activities for postprocessing

> **NOTE:** `data/LVVI3/LVVI3_activities_inference_postprocessing.csv` **no longer exists** (it lived in the deleted LVVI3_YouTube project) and must be **reconstructed before re-run**.

In [8]:
# Get LVVI3 survey activities and new activities discovered from YouTube data for postprocessing 
df_activities_post_origin = pd.read_csv('../../../data/LVVI3/LVVI3_activities_inference_postprocessing.csv')
df_activities_post = df_activities_post_origin[df_activities_post_origin['type']=='survey_activity']
ls_activities_post = df_activities_post['activity'].unique().tolist()
ls_subactivities_post = df_activities_post['sub_activity'].unique().tolist()
ls_activities_post = list(set(ls_activities_post + ls_subactivities_post))
print(ls_activities_post)

['Kayaking', 'Backcountry skiing', 'Other nature observation', 'Nordic walking', 'Bird hunting', 'Snorkelling', 'Frisbee golf', 'Small game hunting', 'Water sports', 'Skiing', 'Skating', 'Nature product collection', 'Cross-country skiing', 'Flying a kite', 'Keeping livestock in a natural environment in leisure time', 'Management of meadows or other natural habitats', 'Fuelwood procurement, collection of twigs, branches and trees for domestic use', 'Worm fishing', 'Walking', 'Snowboarding', 'Canoeing', 'Sledding', 'Swimming', 'Mountain biking', 'Flying a drone', 'Fly fishing', 'Painting', 'Free guided excursion', 'Sailing small boat', 'Outdoor activities with children', 'Scout excursion', 'Travelling or staying in a caravan or camper', 'Downhill skiing', 'Motorboat small boat', 'Guided excursion for a fee', 'Boating', 'Orienteering', 'Guided tour', 'Camping', 'Kicksledding', 'Crabbing', 'Trolling', 'Hunting', 'Fishing with long line, hook etc.', 'Golf', 'Growing ornamental plants, veget

## Match with activities for postprocessing

In [9]:
# Get activities inferenced by LLM from Flickr data
ls_activities_inferenced = df_inferenced.activity.unique().tolist()

In [10]:
ls_activities_missing = [act for act in ls_activities_inferenced if act not in ls_activities_post]
len(ls_activities_missing)

49

In [11]:
non_activities =[
    'Burnout', 'Campfire', 'Airshow', 'Köydenheitto',
    'Tarpoja', 'Solmumestari', 'Day trip', 'Tastings',
    'Backpacking', 'Dancing', 'Cooking with fire',
    'Cooking', 'Knot tying', 
]

df_inferenced = df_inferenced[~df_inferenced["activity"].isin(non_activities)]

In [12]:
# Rename inferenced activities to match with the LVVI3 activity list
df_inferenced["activity"] = df_inferenced["activity"].replace("Underwater photography", "Nature photography")
df_inferenced["activity"] = df_inferenced["activity"].replace("Gravel bicycling", "Cycling")
df_inferenced["activity"] = df_inferenced["activity"].replace("Flying a kite, drone or drone", "Flying a drone")
df_inferenced["activity"] = df_inferenced["activity"].replace("Paddling", "Boating")
df_inferenced["activity"] = df_inferenced["activity"].replace("Birding", "Bird watching")
df_inferenced["activity"] = df_inferenced["activity"].replace("Winter hiking", "Hiking")
df_inferenced["activity"] = df_inferenced["activity"].replace("Mountain climbing", "Climbing")
df_inferenced["activity"] = df_inferenced["activity"].replace("Hike", "Hiking")
df_inferenced["activity"] = df_inferenced["activity"].replace("Fat biking", "Cycling")
df_inferenced["activity"] = df_inferenced["activity"].replace("Vaellus", "Hiking")
df_inferenced["activity"] = df_inferenced["activity"].replace("Sailing", "Boating")
df_inferenced["activity"] = df_inferenced["activity"].replace("Picnics", "Picnicking")
df_inferenced["activity"] = df_inferenced["activity"].replace("Drone flying", "Flying a drone")
df_inferenced["activity"] = df_inferenced["activity"].replace("Winter walking", "Hiking")
df_inferenced["activity"] = df_inferenced["activity"].replace("Trekking", "Hiking")
df_inferenced["activity"] = df_inferenced["activity"].replace("Rafting", "Boating")
df_inferenced["activity"] = df_inferenced["activity"].replace("Sunbathing or lounging elsewhere in nature", "Sunbathing")
df_inferenced["activity"] = df_inferenced["activity"].replace("Watchingsunset", "Nature observation")
df_inferenced["activity"] = df_inferenced["activity"].replace("Randonee walking", "Hiking")
df_inferenced["activity"] = df_inferenced["activity"].replace("Sunbathing, lounging or sightseeing on the beach", "Sunbathing")
df_inferenced["activity"] = df_inferenced["activity"].replace("Hunter skiing", "Skiing")
df_inferenced["activity"] = df_inferenced["activity"].replace("Biking", "Cycling")
df_inferenced["activity"] = df_inferenced["activity"].replace("Stargazing", "Nature observation")
df_inferenced["activity"] = df_inferenced["activity"].replace("Snow walk", "Walking")


In [13]:
# Update activity type to new_activity for those are not in the LVVI3 activity list
df_inferenced["activity"] = df_inferenced["activity"].replace("Grill", "Grilling")
df_inferenced.loc[df_inferenced["activity"] == "Grilling", "type"] = "new_activity"

df_inferenced.loc[df_inferenced["activity"] == "Rallying", "type"] = "new_activity"
df_inferenced.loc[df_inferenced["activity"] == "Ice fishing", "type"] = "new_activity"
df_inferenced.loc[df_inferenced["activity"] == "Yoga", "type"] = "new_activity"
df_inferenced.loc[df_inferenced["activity"] == "Archery", "type"] = "new_activity"
df_inferenced.loc[df_inferenced["activity"] == "Glider flying", "type"] = "new_activity"

In [14]:
# Rename activities and label those relevant to LVVI3 activities as survey_activity
df_inferenced["activity"] = df_inferenced["activity"].replace("Dog sledding", "Husky safari")
df_inferenced.loc[df_inferenced["activity"] == "Husky safari", "type"] = "new_activity"

df_inferenced.loc[df_inferenced["activity"] == "Ice climbing", "type"] = "new_activity"

df_inferenced["activity"] = df_inferenced["activity"].replace("Northern lights viewing", "Northern lights watching")
df_inferenced.loc[df_inferenced["activity"] == "Northern lights watching", "type"] = "new_activity"

df_inferenced.loc[df_inferenced["activity"] == "Reindeer racing", "type"] = "new_activity"

df_inferenced["activity"] = df_inferenced["activity"].replace("Glide", "Glider flying")
df_inferenced.loc[df_inferenced["activity"] == "Glider flying", "type"] = "new_activity"

df_inferenced["activity"] = df_inferenced["activity"].replace("Reindeer herding", "Reindeer safari")
df_inferenced.loc[df_inferenced["activity"] == "Reindeer safari", "type"] = "new_activity"

In [15]:
df_activities_survey = df_inferenced[df_inferenced['type'] == 'survey_activity']
ls_activities_survey = df_activities_survey['activity'].unique().tolist()
ls_activities_survey_not_matching = [activity for activity in ls_activities_survey if activity not in ls_activities_post]
len(ls_activities_survey_not_matching)

0

In [16]:
df_activities_new = df_inferenced[df_inferenced['type'] == 'new_activity']
ls_activities_new = df_activities_new['activity'].unique().tolist()
ls_activities_new_matching = [activity for activity in ls_activities_new if activity in ls_activities_post]
# Update activity type to survey_activity for those are in the LVVI3 activity list
for act in ls_activities_new_matching:
    df_inferenced.loc[df_inferenced["activity"] == act, "type"] = "survey_activity"

In [17]:
df_activities_new = df_inferenced[df_inferenced['type'] == 'new_activity']
ls_activities_new = df_activities_new['activity'].unique().tolist()
len(ls_activities_new)

11

In [18]:
ls_activities_new

['Grilling',
 'Rallying',
 'Husky safari',
 'Ice climbing',
 'Yoga',
 'Ice fishing',
 'Northern lights watching',
 'Reindeer racing',
 'Glider flying',
 'Reindeer safari',
 'Archery']

## Map sub_activities to activities

In [19]:
# Create a mapping: sub_activity → activity
sub_to_activity = df_activities_post_origin.set_index('sub_activity')['activity'].to_dict()

# Rename the activity column to sub_activity
df_inferenced = df_inferenced.rename(columns={"activity": "sub_activity"})
# Map the 'sub_activity' column to get the corresponding 'activity'
df_inferenced['activity'] = df_inferenced['sub_activity'].map(sub_to_activity)

# Update activity type
df_inferenced = df_inferenced.drop(columns=['type'])
df_inferenced = df_inferenced.merge(df_activities_post_origin[['structural_activity', 'sub_activity', 'type']], on='sub_activity', how='left')

In [20]:
df_inferenced.activity.value_counts()

activity
Cycling                                                        294
Walking                                                        286
Skiing                                                         165
Nature photography                                             155
Boating                                                        129
Diving                                                         124
Skating                                                        121
Guided tour                                                     86
Forest                                                          55
Snowmobiling                                                    38
Husky safari                                                    27
Fishing                                                         27
Climbing                                                        25
Camping                                                         23
Flying a drone                                       

In [21]:
df_inferenced.sub_activity.value_counts()

sub_activity
Hiking                  271
Mountain biking         261
Nature photography      155
Cross-country skiing    153
Snowshoeing             108
                       ... 
Reindeer racing           1
Horseback riding          1
Reindeer safari           1
Flying a kite             1
Archery                   1
Name: count, Length: 61, dtype: int64